# DSAA5009 Final - CP05 Training Smoke Test

这个 notebook 用于完成 CP-05：在 Colab 上对 `google/flan-t5-base` 跑通 1 个 batch 的 `forward + backward`，并产出 `results/metrics/training_smoke_test.json`。


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!nvidia-smi


In [ ]:
%cd /content
!rm -rf dsaa-5009-final
!git clone /YOUR_GITHUB_REPO_URL_HERE/ dsaa-5009-final
%cd /content/dsaa-5009-final


In [ ]:
%cd /content/dsaa-5009-final
!python -m pip install --upgrade pip
!pip install -r requirements.txt


In [ ]:
%cd /content/dsaa-5009-final
%%writefile src/training/trainer.py
"""Minimal training utilities for CP-05 smoke test."""

from __future__ import annotations

from dataclasses import dataclass
from typing import Dict, List

import torch
from transformers import PreTrainedTokenizerBase

MAX_INPUT_LENGTH = 512
MAX_TARGET_LENGTH = 128


@dataclass(frozen=True)
class TrainingSmokeConfig:
    max_input_length: int = MAX_INPUT_LENGTH
    max_target_length: int = MAX_TARGET_LENGTH
    batch_size: int = 2


def tokenize_seq2seq_batch(
    samples: List[Dict[str, str]],
    tokenizer: PreTrainedTokenizerBase,
    config: TrainingSmokeConfig = TrainingSmokeConfig(),
) -> Dict[str, torch.Tensor]:
    inputs = [sample['input'] for sample in samples]
    targets = [sample['target'] for sample in samples]

    model_inputs = tokenizer(
        inputs,
        max_length=config.max_input_length,
        truncation=True,
        padding=True,
        return_tensors='pt',
    )

    with tokenizer.as_target_tokenizer():
        labels = tokenizer(
            targets,
            max_length=config.max_target_length,
            truncation=True,
            padding=True,
            return_tensors='pt',
        )

    label_ids = labels['input_ids']
    label_ids[label_ids == tokenizer.pad_token_id] = -100
    model_inputs['labels'] = label_ids
    return model_inputs


def move_batch_to_device(batch: Dict[str, torch.Tensor], device: torch.device) -> Dict[str, torch.Tensor]:
    return {key: value.to(device) for key, value in batch.items()}


def run_single_training_step(model, batch: Dict[str, torch.Tensor], lr: float = 1e-3) -> Dict[str, float]:
    model.train()
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
    optimizer.zero_grad()

    outputs = model(**batch)
    loss = outputs.loss
    loss.backward()
    optimizer.step()

    grad_norm_sq = 0.0
    for param in model.parameters():
        if param.grad is not None:
            grad_norm_sq += float(torch.norm(param.grad.detach(), p=2).item() ** 2)

    return {
        'loss': float(loss.detach().cpu().item()),
        'grad_norm': grad_norm_sq ** 0.5,
    }


In [ ]:
%cd /content/dsaa-5009-final
%%writefile scripts/check_training_step.py
#!/usr/bin/env python3
"""CP-05 smoke test: tokenize a tiny batch and run 1 forward/backward step."""

from __future__ import annotations

import json
from pathlib import Path

import torch
from datasets import load_dataset

from src.data.preprocessing import build_multitask_samples
from src.models.load_model import ModelConfig, prepare_model
from src.training.trainer import (
    TrainingSmokeConfig,
    move_batch_to_device,
    run_single_training_step,
    tokenize_seq2seq_batch,
)

ROOT = Path(__file__).resolve().parents[1]
METRICS_DIR = ROOT / 'results' / 'metrics'
METRICS_DIR.mkdir(parents=True, exist_ok=True)


def main() -> None:
    smoke_config = TrainingSmokeConfig(batch_size=2)

    dataset = load_dataset('knkarthick/dialogsum', split='train[:2]')
    raw_samples = [dataset[i] for i in range(len(dataset))]

    summarize_samples = []
    for sample in raw_samples:
        multitask_samples = build_multitask_samples(sample)
        summarize_samples.append(multitask_samples[0])

    tokenizer, model, model_report = prepare_model(ModelConfig(model_name='google/flan-t5-base'))

    batch = tokenize_seq2seq_batch(summarize_samples, tokenizer, smoke_config)
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = model.to(device)
    batch = move_batch_to_device(batch, device)

    step_report = run_single_training_step(model, batch)

    report = {
        'model': 'google/flan-t5-base',
        'device': str(device),
        'batch_size': smoke_config.batch_size,
        'input_shape': list(batch['input_ids'].shape),
        'label_shape': list(batch['labels'].shape),
        'loss': step_report['loss'],
        'grad_norm': step_report['grad_norm'],
        'loss_is_finite': bool(torch.isfinite(torch.tensor(step_report['loss'])).item()),
        'used_samples': [
            {
                'input_preview': sample['input'][:120],
                'target_preview': sample['target'][:120],
                'length_bucket': sample['length_bucket'],
            }
            for sample in summarize_samples
        ],
        'model_report': {
            'added_tokens': model_report['added_tokens'],
            'trainable_ratio': model_report['trainable_ratio'],
        },
        'verification': {
            'forward_backward_ok': True,
            'loss_non_nan': bool(torch.isfinite(torch.tensor(step_report['loss'])).item()),
            'batch_tokenized': True,
        },
    }

    output_path = METRICS_DIR / 'training_smoke_test.json'
    output_path.write_text(json.dumps(report, indent=2, ensure_ascii=False))

    print('Training smoke test passed.')
    print(f"Device: {device}")
    print(f"Loss: {report['loss']:.6f}")
    print(f"Saved report to: {output_path}")


if __name__ == '__main__':
    main()


In [ ]:
%cd /content/dsaa-5009-final
!PYTHONPATH=. python scripts/check_training_step.py


In [ ]:
%cd /content/dsaa-5009-final
!cat results/metrics/training_smoke_test.json
